## 🏗️ Tipos de Tablas en Databricks

Una vez comprendido cómo funciona una Delta Table y cómo Unity Catalog gobierna los objetos del Lakehouse, es importante conocer los dos tipos principales de tablas que podemos definir en Databricks.

La principal diferencia entre ellas radica en quién administra los datos físicos almacenados. Estas son:

---

* #### 📦 Managed Tables (Tablas Administradas)

Las tablas administradas son aquellas donde Databricks controla tanto:

* ✅ Los metadatos registrados en Unity Catalog
* ✅ Los archivos físicos almacenados en Delta Lake

En este modelo, Databricks se encarga completamente de la administración de la tabla. Por otro lado, la eliminación de este tipo de tablas conlleva a eliminar:

* ❌ Los metadatos
* ❌ Los archivos físicos asociados

Por esta razón, las Managed Tables son una excelente opción cuando deseamos que Databricks administre el ciclo de vida completo de los datos.

#### Creación de una Managed Table

```sql id="3kj5qu"
CREATE TABLE catalog.schema.table
(
    ...
)
USING DELTA;
```

📌 En este caso utilizamos Delta Lake como formato de almacenamiento nativo.

---

* #### 🌎 External Tables (Tablas Externas)

Las tablas externas funcionan de manera diferente, donde Databricks únicamente administra:

* ✅ Los metadatos registrados en Unity Catalog

Mientras que los datos físicos permanecen almacenados en una ubicación externa, por ejemplo:

* ☁️ Amazon S3
* ☁️ Azure Data Lake Storage (ADLS)
* ☁️ Google Cloud Storage (GCS)

Y si son eliminadas, Databricks elimina:

* ❌ Los metadatos

Pero conserva:

* ✅ Los archivos físicos en el almacenamiento externo

Esto permite que los datos continúen existiendo incluso después de eliminar la definición de la tabla.

---

##### 🔐 Requisitos para trabajar con External Tables

Antes de crear una tabla externa, Unity Catalog necesita conocer:

* 1️⃣ Cómo autenticarse contra el almacenamiento externo
* 2️⃣ Dónde se encuentran físicamente los datos

Para ello se utilizan dos objetos fundamentales:

* **Storage Credential**: Define las credenciales que permitirán acceder al almacenamiento externo.
    ```sql id="79azml"
    CREATE STORAGE CREDENTIAL nombre_credential
    ...
    ```

* **External Location**: Asocia una ruta física con las credenciales definidas previamente.
    ```sql id="m6w7ho"
    CREATE EXTERNAL LOCATION nombre_location
    URL 'ruta_externa'
    WITH (
        STORAGE CREDENTIAL nombre_credential
    );
    ```
---
##### 🚀 Creación de una External Table

Una vez configurados los objetos anteriores, podemos registrar la tabla:
```sql id="8afn63"
CREATE TABLE catalog.schema.table
USING DELTA
LOCATION 'ruta_externa';
```

* 📌 Los metadatos quedarán registrados en Unity Catalog.
* 📌 Los datos físicos permanecerán en la ubicación externa configurada.

---
#### 🎯 Conclusión

💡 La decisión entre Managed y External no depende del formato de los datos, sino de quién será responsable de administrar el almacenamiento físico de la información.


### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("06TablesTypes").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")

#### 🔍 ¿Qué acaba de ocurrir?

Acabamos de crear una Spark Session. Esta sesión representa nuestro punto de entrada hacia:
* 🏗️ Apache Spark
* 🏗️ Databricks Runtime
* 🏗️ Delta Lake
* 🏗️ Unity Catalog

### ⚡ Managed Table

- Nombre: `catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table_managed`

In [0]:
spark.sql("""
          CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table_managed
          (
              customer_id INT,
              customer_name VARCHAR(50),
              customer_country VARCHAR(30)
          )
          USING DELTA;
          """)
print("Tabla creada correctamente")

In [0]:
## VERIFICAMOS SU ESRUCTURA Y TIPO
display(spark.sql("DESCRIBE EXTENDED catalog_databricks_2026_de.schema_databricks_2026_de.customers_delta_table_managed"))
### Resultado ⬇️: Type - MANAGED 

### 📂 External Table

- Nombre: `catalog_databricks_2026_de.schema_databricks_2026_de.tips_external_table`

- Asimismo, debemos realizar lo siguiente:
[pasos-previos-external-tables.md](https://github.com/BrayanR03/Databricks-DE-2026/tree/main/assets/pasos-previos-external-tables.md)

In [0]:
spark.sql("""
        CREATE TABLE catalog_databricks_2026_de.schema_databricks_2026_de.tips_external_table
        USING CSV
        OPTIONS (
        header = 'true',
        inferSchema = 'true',
        delimiter = ','
        )
        LOCATION 's3://bucket-datasets-brayan/tips.csv';
          """)

"""
  💡 Debemos tener en cuenta que las opciones para cada tipo de archivo en las
     External Tables, va a depender del tipo de archivo origen. Además, podemos
     inferir automáticamente su esquema o lo podemos definir previamente para un
     mayor control.
"""


print("Tabla externa creada correctamente")

In [0]:
## VERIFICAMOS SU ESRUCTURA Y TIPO
display(spark.sql("DESCRIBE EXTENDED catalog_databricks_2026_de.schema_databricks_2026_de.tips_external_table"))
### Resultado ⬇️: Type - EXTERNAL 